# Genre-Aware Popularity Recommender
## Spotify MPD — filtered playlists (genre-labeled, 50+ tracks)

Pipeline:
1. Load filtered playlists and their tracks
2. Assign each track a genre by majority vote across playlists it appears in
3. Build per-genre popularity rankings
4. Given a playlist seed, infer its genre and recommend top-K popular tracks from that genre
5. Evaluate with Precision@K, Recall@K and genre inference accuracy

---
## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROCESSED = Path('../processed')
OUT       = Path('outputs')
OUT.mkdir(exist_ok=True)

SEED_RATIO = 0.8
SAMPLE_N   = 25     # tracks sampled from seed for genre inference
K_VALUES   = [1, 5, 10, 20]
MAX_EVAL   = 25000
RANDOM_SEED = 42

---
## 1. Load Data

In [ ]:
filtered = pd.read_parquet(PROCESSED / 'filtered_playlists.parquet')
trks     = pd.read_parquet(PROCESSED / 'tracks.parquet',
                           columns=['pid', 'pos', 'track_uri', 'track_name', 'artist_name'])

# Keep only tracks from filtered playlists
valid_pids = set(filtered['pid'])
trks = trks[trks['pid'].isin(valid_pids)].reset_index(drop=True)

print(f'Filtered playlists : {len(filtered):,}')
print(f'Track entries      : {len(trks):,}')
print(f'Unique tracks      : {trks["track_uri"].nunique():,}')
print(f'\nGenre breakdown:')
print(filtered['genre'].value_counts().to_string())

---
## 2. Train / Test Split

Split at the playlist level — 80% train, 10% val, 10% test. Stratified by genre to preserve genre distribution across splits.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

train_pids, val_pids, test_pids = [], [], []

for genre, grp in filtered.groupby('genre'):
    idx = grp['pid'].values.copy()
    rng.shuffle(idx)
    n      = len(idx)
    n_test = max(1, round(n * 0.1))
    n_val  = max(1, round(n * 0.1))
    test_pids.extend(idx[:n_test])
    val_pids .extend(idx[n_test:n_test + n_val])
    train_pids.extend(idx[n_test + n_val:])

train_pids = set(train_pids)
val_pids   = set(val_pids)
test_pids  = set(test_pids)

print(f'Train playlists: {len(train_pids):,}')
print(f'Val   playlists: {len(val_pids):,}')
print(f'Test  playlists: {len(test_pids):,}')

---
## 3. Build Track Genre Mapping

For each track, count how many times it appears in each genre's playlists (training set only). Assign the majority genre.

In [ ]:
pid2genre = dict(zip(filtered['pid'], filtered['genre']))

train_trks = trks[trks['pid'].isin(train_pids)].copy()
train_trks['genre'] = train_trks['pid'].map(pid2genre)

# Count (track_uri, genre) co-occurrences
cooc = (
    train_trks.groupby(['track_uri', 'genre'])
    .size()
    .reset_index(name='count')
)

# Majority vote — assign each track its most frequent genre
track_genres = (
    cooc.sort_values('count', ascending=False)
    .drop_duplicates('track_uri')
    .rename(columns={'count': 'genre_count'})
    .reset_index(drop=True)
)
track_genre_map = dict(zip(track_genres['track_uri'], track_genres['genre']))

# Per-genre popularity ranking (for recommendations)
meta = trks.drop_duplicates('track_uri')[['track_uri', 'track_name', 'artist_name']]
genre_popularity = (
    cooc.merge(meta, on='track_uri', how='left')
    .sort_values(['genre', 'count'], ascending=[True, False])
    .reset_index(drop=True)
)

print(f'Tracks with genre assigned: {len(track_genres):,}')
print(f'\nTracks per genre (majority vote):')
print(track_genres['genre'].value_counts().to_string())

---
## 4. Recommender

Given a playlist seed:
1. Randomly sample up to `SAMPLE_N` tracks
2. Look up their genres and take the majority vote
3. Return top-K most popular tracks from that genre not already in the playlist

In [ ]:
# Build genre -> ordered track list lookup
genre_tracks = {}
for genre, grp in genre_popularity.groupby('genre'):
    genre_tracks[genre] = list(
        zip(grp['track_uri'], grp['track_name'], grp['artist_name'])
    )

def infer_genre(track_uris):
    genres = [track_genre_map[u] for u in track_uris if u in track_genre_map]
    if not genres:
        return 'unknown'
    return Counter(genres).most_common(1)[0][0]

def recommend(playlist_tracks, k=10):
    genre = infer_genre(playlist_tracks)
    if genre == 'unknown' or genre not in genre_tracks:
        return genre, []
    seen = set(playlist_tracks)
    recs = []
    for track_uri, track_name, artist_name in genre_tracks[genre]:
        if track_uri not in seen:
            recs.append({'track_uri': track_uri, 'track_name': track_name, 'artist_name': artist_name})
        if len(recs) >= k:
            break
    return genre, recs

print('Recommender ready.')

### 4.1 Demo — single playlist

In [ ]:
sample_pid  = list(test_pids)[0]
pl_tracks   = (
    trks[trks['pid'] == sample_pid]
    .sort_values('pos')['track_uri']
    .tolist()
)
true_genre  = pid2genre[sample_pid]
sp          = max(1, int(len(pl_tracks) * SEED_RATIO))
seed        = pl_tracks[:sp]

inferred, recs = recommend(seed, k=10)

print(f'Playlist pid   : {sample_pid}')
print(f'True genre     : {true_genre}')
print(f'Inferred genre : {inferred}')
print(f'\nTop-10 recommendations:')
for i, r in enumerate(recs, 1):
    print(f'  {i:2d}. {r["track_name"][:35]:35s}  –  {r["artist_name"]}')

---
## 5. Evaluation

For each test playlist:
- Seed = first 80% of tracks
- Holdout = remaining 20%
- Recommend top-K from seed, compare with holdout

Metrics:
- **Precision@K** — fraction of recommendations that appear in the holdout
- **Recall@K** — fraction of holdout tracks that appear in the top-K recommendations
- **Genre accuracy** — how often the inferred genre matches the playlist's true genre label

In [ ]:
rng     = np.random.default_rng(RANDOM_SEED)
test_pl = list(test_pids)
if len(test_pl) > MAX_EVAL:
    test_pl = list(rng.choice(test_pl, MAX_EVAL, replace=False))

metrics = {f'Prec@{k}': [] for k in K_VALUES}
metrics.update({f'Recall@{k}': [] for k in K_VALUES})
genre_hits = []

for pid in test_pl:
    pl = (
        trks[trks['pid'] == pid]
        .sort_values('pos')['track_uri']
        .tolist()
    )
    if len(pl) < 5:
        continue

    sp      = max(1, int(len(pl) * SEED_RATIO))
    seed    = pl[:sp]
    holdout = set(pl[sp:])
    if not holdout:
        continue

    inferred, recs = recommend(seed, k=max(K_VALUES))
    rec_uris = [r['track_uri'] for r in recs]

    genre_hits.append(1 if inferred == pid2genre.get(pid) else 0)

    for k in K_VALUES:
        hits = len(set(rec_uris[:k]) & holdout)
        metrics[f'Prec@{k}'].append(hits / k)
        metrics[f'Recall@{k}'].append(hits / len(holdout))

results = {m: np.mean(v) for m, v in metrics.items()}
results['Genre Accuracy'] = np.mean(genre_hits)

results_df = pd.DataFrame([results]).T.rename(columns={0: 'Score'})
print(f'Evaluated on {len(genre_hits):,} playlists\n')
print(results_df.to_string(float_format='%.4f'))

---
## 6. Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Precision@K
prec_vals = [results[f'Prec@{k}'] for k in K_VALUES]
axes[0].bar([str(k) for k in K_VALUES], prec_vals, color='steelblue')
axes[0].set_title('Precision@K', fontweight='bold')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Score')

# Recall@K
rec_vals = [results[f'Recall@{k}'] for k in K_VALUES]
axes[1].bar([str(k) for k in K_VALUES], rec_vals, color='coral')
axes[1].set_title('Recall@K', fontweight='bold')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Score')

plt.suptitle('Genre-Aware Popularity Recommender', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'evaluation_results.png', bbox_inches='tight')
plt.show()

print(f'Genre inference accuracy: {results["Genre Accuracy"]:.4f}')

---
## 7. Per-genre Breakdown

In [ ]:
genre_metrics = {}

for pid in test_pl:
    pl = (
        trks[trks['pid'] == pid]
        .sort_values('pos')['track_uri']
        .tolist()
    )
    if len(pl) < 5:
        continue

    sp      = max(1, int(len(pl) * SEED_RATIO))
    seed    = pl[:sp]
    holdout = set(pl[sp:])
    if not holdout:
        continue

    true_genre     = pid2genre.get(pid, 'unknown')
    inferred, recs = recommend(seed, k=10)
    rec_uris       = [r['track_uri'] for r in recs]
    hits           = len(set(rec_uris[:10]) & holdout)

    if true_genre not in genre_metrics:
        genre_metrics[true_genre] = {'Prec@10': [], 'Recall@10': []}
    genre_metrics[true_genre]['Prec@10'].append(hits / 10)
    genre_metrics[true_genre]['Recall@10'].append(hits / len(holdout))

genre_summary = pd.DataFrame([
    {
        'genre':      g,
        'Prec@10':    np.mean(v['Prec@10']),
        'Recall@10':  np.mean(v['Recall@10']),
        'n_playlists': len(v['Prec@10']),
    }
    for g, v in genre_metrics.items()
]).sort_values('Recall@10', ascending=False).reset_index(drop=True)

print(genre_summary.to_string(index=False, float_format='%.4f'))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(genre_summary))
ax.bar([i - 0.2 for i in x], genre_summary['Prec@10'],   width=0.4, label='Prec@10',   color='steelblue')
ax.bar([i + 0.2 for i in x], genre_summary['Recall@10'], width=0.4, label='Recall@10', color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(genre_summary['genre'], rotation=30, ha='right')
ax.set_title('Per-genre Precision@10 and Recall@10', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUT / 'evaluation_per_genre.png', bbox_inches='tight')
plt.show()

---
## 8. Summary

The genre-aware popularity recommender works as follows:
- Given a playlist, it infers the dominant genre from a random sample of tracks
- It then recommends the most popular tracks from that genre not already in the playlist
- Genre inference accuracy tells us how reliable the genre signal from playlist names is
- Precision@K and Recall@K measure recommendation quality against the held-out tracks

This approach is justified by the EDA findings:
- Playlist names carry strong genre signal (top names: chill, country, rap, workout)
- Track popularity follows a power-law — popular tracks within a genre are strong candidates
- No audio features or external APIs are available, making playlist name the only genre proxy